# imports

In [9]:
%reload_ext autoreload
%autoreload 2

import torch
import os
import json
from tqdm import tqdm
import numpy as np
import scipy.stats as stats

In [2]:
import pandas as pd

df = pd.read_csv('/Users/arubique/github/sort-and-search/results/accs_imagenet-1k_model_eval_350_sum_42.csv')
df.head()


,Unnamed: 0,GT,Optimal_global,Optimal_sample,Uniform_global_10,Uniform_sample_10,Random_global_10,Random_sample_10,Uniform_global_50,Uniform_sample_50,Random_global_50,Random_sample_50,Uniform_global_100,Uniform_sample_100,Random_global_100,Random_sample_100,Uniform_global_1000,Uniform_sample_1000,Random_global_1000,Random_sample_1000
0,0,92.9,92.9,99.95,95.0,95.0,92.8,92.8,91.0,99.0,94.70,99.85,91.5,99.5,89.7,98.35,93.65,99.95,92.55,99.95
1,1,89.2,89.2,99.95,95.0,95.0,80.5,92.8,91.0,99.0,85.00,99.85,90.5,99.5,82.5,98.35,89.45,99.95,89.90,99.95
2,2,90.4,90.4,99.95,95.0,95.0,92.8,92.8,97.0,99.0,88.70,99.85,91.5,99.5,92.4,98.35,90.15,99.95,90.30,99.95
3,3,88.1,88.1,99.95,85.0,95.0,92.8,92.8,93.0,99.0,88.70,99.85,92.5,99.5,89.7,98.35,87.85,99.95,87.65,99.95
4,4,90.7,90.7,99.95,95.0,95.0,92.8,92.8,91.0,99.0,92.35,99.85,90.5,99.5,88.9,98.35,90.35,99.95,90.75,99.95


In [6]:
# gt_scores = []
# accs_per_method = {}
# for row in df.iterrows():
#     print(row)
#     gt_scores.append(row['GT'])
#     for key, value in row.items():
#         if key != 'GT':
#             accs_per_method[key].append(value)
    # rank_corrs = []
    # # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
    # pred_accs_as_np = np.stack(list(accs.values()), axis=0)
    # maes = np.abs(
    #     pred_accs_as_np - gt_scores[0][:, None]
    # )
    # for i in range(pred_accs_as_np.shape[1]):
    #     rank_corrs.append(safe_spearmanr(
    #         pred_accs_as_np[:, i],
    #         gt_scores[0][:, None],
    #     ))
    # maes_per_method[method_name] = maes
    # rank_corrs_per_method[method_name] = np.array(rank_corrs)

accs_per_method = {}
for column in df.columns:
    accs_per_method[column] = df[column]

In [12]:
STD_EPS = 1e-9


def safe_spearmanr(x_data, y_data):
    """
    Compute Spearman correlation with safe handling of constant arrays.

    Args:
        x_data: First array for correlation
        y_data: Second array for correlation

    Returns:
        float: Spearman correlation coefficient, or np.nan if either array is constant
    """
    # Check if either array is constant (all values are the same)
    if np.std(x_data) < STD_EPS or np.std(y_data) < STD_EPS:
        return np.nan  # or 0, depending on your preference
    else:
        return stats.spearmanr(x_data, y_data).statistic

In [ ]:
accs_per_method

In [21]:
gt_scores = np.array(list(accs_per_method['GT']))
print(gt_scores)
maes_per_method = {}
rank_corrs_per_method = {}
for method_name, accs in accs_per_method.items():
    if method_name in ['GT', 'Unnamed: 0']:
        continue
    rank_corrs = []
    # pred_accs_as_np = np.stack(list(predicted_accs.values()), axis=0)
    pred_accs_as_np = np.stack(list(accs), axis=0)
    maes = np.abs(
        pred_accs_as_np - gt_scores
    )
    # for i in range(pred_accs_as_np):
    rank_corrs.append(safe_spearmanr(
        pred_accs_as_np,
        gt_scores,
    ))
    maes_per_method[method_name] = maes
    rank_corrs_per_method[method_name] = np.array(rank_corrs)

[92.9  89.2  90.4  88.1  90.7  85.1  80.85 88.3  90.4  91.15 85.6  87.7
 81.4  83.65 87.8  91.65 86.05 87.6  89.65 91.05 89.05 90.   87.55 90.25
 88.05 91.3  92.75 82.65 89.5  89.2  90.05 92.45 90.25 91.15 90.8  91.55
 89.5  86.85 90.1  84.9  85.15 85.5  91.65 91.35 92.5  81.25 91.65 91.45
 90.65 86.7 ]


In [24]:
for method_name, maes in maes_per_method.items():
    print(method_name, maes.mean())

Optimal_global 0.0
Optimal_sample 11.25
Uniform_global_10 5.9239999999999995
Uniform_sample_10 6.300000000000001
Random_global_10 4.297999999999997
Random_sample_10 4.1039999999999965
Uniform_global_50 3.244000000000001
Uniform_sample_50 10.3
Random_global_50 3.0349999999999993
Random_sample_50 11.149999999999993
Uniform_global_100 1.6999999999999997
Uniform_sample_100 10.8
Random_global_100 2.877000000000001
Random_sample_100 9.649999999999995
Uniform_global_1000 0.38599999999999995
Uniform_sample_1000 11.25
Random_global_1000 0.5380000000000018
Random_sample_1000 11.25


In [23]:
rank_corrs_per_method

{'Optimal_global': array([1.]),
 'Optimal_sample': array([nan]),
 'Uniform_global_10': array([0.44479552]),
 'Uniform_sample_10': array([nan]),
 'Random_global_10': array([0.3342052]),
 'Random_sample_10': array([nan]),
 'Uniform_global_50': array([0.42937651]),
 'Uniform_sample_50': array([nan]),
 'Random_global_50': array([0.54614904]),
 'Random_sample_50': array([nan]),
 'Uniform_global_100': array([0.7417297]),
 'Uniform_sample_100': array([nan]),
 'Random_global_100': array([0.74574845]),
 'Random_sample_100': array([nan]),
 'Uniform_global_1000': array([0.9804953]),
 'Uniform_sample_1000': array([nan]),
 'Random_global_1000': array([0.96968025]),
 'Random_sample_1000': array([nan])}